# İHA Video Arama — AU-AIR denemesi (Kaggle)

`poc/kaggle_pipeline_trial.ipynb`'nin sadeleştirilmiş hali — **sadece AU-AIR**
üzerinden gidiyoruz, SeaDroneSee/genel video akışı (kalibrasyon, toplu yükleme,
manifest.json karşılaştırması, tekne sorguları, vLLM) bu deftere alınmadı.

Amaç: "AU-AIR bizim kullanım amacımıza uyar mı" sorusunu **gerçek GPU'da**, 8
videonun TAMAMIYLA cevaplamak — `poc/auair_adapter.py` / `auair_build_videos.py`
/ `auair_ingest.py`'nin (production kodu değiştirilmeden, bkz. o dosyaların
docstring'leri) Kaggle'a taşınmış hali.

**Lisans:** AU-AIR CC BY-NC-SA/CC BY-NC (ticari/savunma kullanımına kapalı,
Qwen3-VL-Embedding-2B'nin VideoCLIP-XL/EBind'i elediği kararla aynı kategori)
— bu **sadece iç doğrulama** denemesi, kalıcı bir ingest yolu değil.

**SONUÇ VİDEOLAR GERÇEK ÇEKİM DEĞİL:** AU-AIR'in ayrık kareleri (~5 FPS, gerçek
zaman damgalarına sadık şekilde birleştirilmiş) kullanılıyor — bu MEKANİZMA
testi (ingest/pencereleme/embedding gerçekten çalışıyor mu), retrieval
KALİTESİ testi değil.

**Ayrı Qdrant koleksiyonu** (`clips_auair_test`) kullanılıyor.

Genel amaçlı (kendi videonuzu/SeaDroneSee'yi test etmek istiyorsanız) defter
için: [poc/kaggle_pipeline_trial.ipynb](kaggle_pipeline_trial.ipynb).

## Kullanım kuralları

- **Hücreleri sırayla çalıştırın.**
- **Ortam değişkenini `set_env(...)` ile değiştirin**, doğrudan `os.environ`
  ile değil (aşağıda tanımlı).
- **Qdrant istemcisini `close()` etmeyin** - önbellekli, gömülü mod dosya
  kilidi kullanıyor.
- Notebook'u **GitHub'dan taze açtığınızdan emin olun** - repo'yu güncellemek
  açık bir Kaggle sekmesindeki hücreleri otomatik güncellemez.
- Önce **Settings → Internet**'in açık olduğundan emin olun (Google Drive'dan
  indirme + git clone gerekiyor).

## 1. GPU + ortam kontrolü

**Önce:** sağ paneldeki **Settings → Accelerator**'dan bir GPU seçin.
**GPU T4 x2** önerilir (P100 değil) - T4'ün davranışını (bf16 yok, fp16'ya
geçiş) Colab'da doğruladık, P100 (Pascal, compute 6.0) farklı/daha eski bir
mimari ve AWQ kuantize modellerle sorunlu olabilir. "x2" yazsa da bu defter
tek GPU kullanıyor (çoklu-GPU paralelliği kurulmadı) - ikinci GPU boşta kalır.

GPU seçtikten sonra **Settings → Internet**'i açın ve oturumu yeniden
başlatın.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!free -g | head -2
!df -h /kaggle/working | tail -1

## 2. Depo + bağımlılıklar

Kaggle'da torch zaten CUDA'lı geldiği için `requirements.txt`'i olduğu gibi
kurmuyoruz (mevcut torch'u bozabilir). Sadece eksikleri kuruyoruz.

`git clone` başarısız olursa: yukarıdaki Internet ayarını kontrol edin.

In [ ]:
import pathlib, subprocess, sys

REPO = pathlib.Path('/kaggle/working/VideoAnalysis')

# Sessiz git komutlari kullanmiyoruz: pull sessizce basarisiz olursa ESKI
# KOD calismaya devam eder ve hata cok sonra alakasiz bir yerde patlar.
if REPO.exists():
    print('Depo mevcut, uzak surumle esitleniyor...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
    print(subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO,
                         capture_output=True, text=True).stdout.strip())
else:
    r = subprocess.run(['git', 'clone',
                        'https://github.com/ykyking1/VideoAnalysis.git', str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

head = subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('\nCalisan surum:', head)

config_src = (REPO / 'common' / 'config.py').read_text(encoding='utf-8')
assert 'LOCAL_STORAGE_PATH' in config_src, (
    'ESKI KOD! git pull calismamis - yukaridaki "Calisan surum" satirini kontrol edin.')

%cd /kaggle/working/VideoAnalysis

# Kaggle'in CUDA'li torch'una DOKUNMUYORUZ - sadece eksik paketler.
# qwen-vl-utils>=0.0.14 kritik: eskisi SESSIZCE bozuk embedding uretiyor.
!pip install -q "transformers>=4.57" "qwen-vl-utils>=0.0.14" accelerate \
    qdrant-client ultralytics opencv-python-headless temporalio \
    pysolar shapely gdown 2>&1 | tail -3

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Kod guncel, hazir. Sonraki hucrede ortam degiskenleri ayarlanacak.')

## 3. Ortam yapılandırması

Kaggle'da Docker yok. İki servisi Docker'sız çalıştırıyoruz:

- **Qdrant** → gömülü mod (`QDRANT_LOCAL_PATH`)
- **Nesne deposu** → yerel dizin (`LOCAL_STORAGE_PATH`), MinIO'ya gerek yok

`QDRANT_COLLECTION` burada baştan `clips_auair_test` — bu defterde başka
koleksiyon kullanmıyoruz.

In [ ]:
import os, sys, importlib

def set_env(**kwargs):
    """Ortam degiskenini ayarlar VE common.config'i yeniden yukler.

    common/config.py env'i IMPORT ANINDA okuyor. Bir degiskeni sonradan
    degistirirseniz, config zaten yuklenmis oldugu icin surec-ici cagrilar
    ESKI degeri gorur. Bu yardimci o tuzagi kapatiyor - notebook boyunca
    env degistirmek icin hep bunu kullanin, dogrudan os.environ'a yazmayin."""
    for k, v in kwargs.items():
        os.environ[k] = str(v)
    if 'common.config' in sys.modules:
        importlib.reload(sys.modules['common.config'])

set_env(
    QDRANT_LOCAL_PATH='/kaggle/working/qdrant_data',
    LOCAL_STORAGE_PATH='/kaggle/working/storage',
    QDRANT_COLLECTION='clips_auair_test',        # AYRI koleksiyon - gercek/SeaDroneSee ile karismasin
    EMBEDDING_BATCH_SIZE=8,                      # T4 16GB icin baslangic
    EMBEDDING_DTYPE='auto',                      # T4 (compute 7.5) -> fp16
    CAPTION_ENABLED='false',                     # vLLM bu defterde yok
)

from common import config
from common.minio_client import backend_name
assert config.LOCAL_STORAGE_PATH, 'LOCAL_STORAGE_PATH okunmadi'
print('Nesne deposu :', backend_name())
print('Qdrant       : gomulu ->', config.QDRANT_LOCAL_PATH)
print('Koleksiyon   :', config.QDRANT_COLLECTION)
print('Batch        :', config.EMBEDDING_BATCH_SIZE)

!python -m scripts.init_storage --skip-postgres

## 4. Ortam doğrulaması

`torch ... CPU-only` ya da `qwen-vl-utils < 0.0.14` görürseniz **durun** —
ikisi de çökmeden sessizce bozuyor.

In [ ]:
!python -m scripts.check_env

## 5. AU-AIR verisini indir

Google Drive'dan iki dosya çekiliyor (idempotent - zaten indirilip açılmışsa
atlanır, oturum kesilip yeniden başlarsa baştan indirmez):

- annotations (~3.9MB, 32.823 kayıt)
- images (~2.2GB, 32.823 JPG)

In [ ]:
import pathlib, zipfile, json

AUAIR_DIR = pathlib.Path('/kaggle/working/auair_data')
AUAIR_DIR.mkdir(exist_ok=True)
ANNOT_ZIP = AUAIR_DIR / 'annotations.zip'
ANNOT_JSON = AUAIR_DIR / 'annotations.json'
IMAGES_ZIP = AUAIR_DIR / 'images.zip'
IMAGES_DIR = AUAIR_DIR / 'images'

if not ANNOT_JSON.exists():
    print('Annotations indiriliyor...')
    !python -m gdown 1boGF0L6olGe_Nu7rd1R8N7YmQErCb0xA -O {ANNOT_ZIP}
    with zipfile.ZipFile(ANNOT_ZIP) as z:
        z.extractall(AUAIR_DIR)
    # Zip icindeki gercek dosya adi degisebilir - annotations.json'a normalize et
    found = [p for p in AUAIR_DIR.rglob('*.json')]
    assert found, 'annotations.json zip icinde bulunamadi'
    if found[0] != ANNOT_JSON:
        found[0].rename(ANNOT_JSON)
    print('Annotations hazir:', ANNOT_JSON)
else:
    print('Annotations zaten mevcut, atlaniyor.')

if not IMAGES_DIR.exists() or not any(IMAGES_DIR.iterdir()):
    print('Goruntuler indiriliyor (2.2GB, biraz surer)...')
    !python -m gdown 1pJ3xfKtHiTdysX5G3dxqKTdGESOBYCxJ -O {IMAGES_ZIP}
    with zipfile.ZipFile(IMAGES_ZIP) as z:
        z.extractall(AUAIR_DIR)
    print('Goruntuler hazir:', IMAGES_DIR)
else:
    print('Goruntuler zaten mevcut, atlaniyor.')

n_annot = len(json.loads(ANNOT_JSON.read_text(encoding='utf-8'))['annotations'])
n_images = len(list(IMAGES_DIR.rglob('*.jpg')))
print(f'\n{n_annot} annotasyon, {n_images} goruntu dosyasi')
assert n_images >= n_annot, (
    f'Goruntu sayisi ({n_images}) annotasyon sayisindan ({n_annot}) az - indirme eksik olabilir.')

## 6. 8 videoyu inşa et

AU-AIR ayrık kareler (~5 FPS, bazı videolarda 15-52sn'lik gerçek boşluklar)
olarak geliyor - ffmpeg concat demuxer ile, kareler arası GERÇEK zaman
damgasına sadık kalarak birleştiriliyor (bkz. `poc/auair_build_videos.py`
docstring'i - sabit FPS varsayımı bu boşlukları sessizce yutardı).

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/VideoAnalysis')
from poc.auair_adapter import load_auair_records, split_by_source_video
from poc.auair_build_videos import build_video

records = load_auair_records(str(ANNOT_JSON))
groups = split_by_source_video(records)
print(f'{len(groups)} kaynak video, {len(records)} toplam kayit')

AUAIR_VIDEOS_DIR = AUAIR_DIR / 'videos'
AUAIR_VIDEOS_DIR.mkdir(exist_ok=True)

for prefix, group in sorted(groups.items()):
    out_path = AUAIR_VIDEOS_DIR / f'{prefix}.mp4'
    if out_path.exists():
        print(f'{prefix}: zaten var, atlaniyor')
        continue
    print(f'{prefix}: {len(group)} kare -> {out_path.name} ...')
    build_video(prefix, group, AUAIR_DIR / 'images', out_path)
    size_mb = out_path.stat().st_size / 1024**2
    print(f'  tamam: {size_mb:.1f} MB')

built = sorted(AUAIR_VIDEOS_DIR.glob('*.mp4'))
print(f'\n{len(built)}/8 video hazir: {[p.name for p in built]}')
assert len(built) == 8, 'Eksik video var - yukaridaki hatalari kontrol edin'

## 7. Embed + ingest (8 videonun tamamı)

Gerçek pipeline: proxy üretimi, AU-AIR telemetri adaptörü (MAVLink DEĞİL,
bkz. `poc/auair_adapter.py`), Qwen3-VL-Embedding-2B ile klip embedding,
YOLO26 görsel alanlar, Qdrant'a yazım (`clips_auair_test`). Caption
atlanıyor (vLLM bu defterde yok).

Yerel makinede (zayıf GPU) tek video (156sn) 27 dakika sürmüştü (0.10x
gerçek-zaman) - 8 videonun tamamı (~2,13 saat) o hızla ~21 saat sürerdi,
bu yüzden Kaggle GPU'suna taşındı. Aşağıdaki hücrenin sonunda çıkan
**agrege gerçek-zaman katsayısı** proje-ozeti.md §8'in 40x varsayımıyla
karşılaştırılacak en değerli sayı.

In [ ]:
from poc.auair_ingest import ingest_one

auair_results = []
for prefix in sorted(groups):
    r = await ingest_one(prefix, groups[prefix], AUAIR_VIDEOS_DIR, skip_caption=True)
    auair_results.append(r)

print(f"\n{'='*70}\nAU-AIR OZET ({len(auair_results)}/8 video)\n{'='*70}")
print(f"{'video':30}{'pencere':>9}{'sure(s)':>10}{'gercek-zaman':>14}{'arac':>7}")
total_dur = total_elapsed = 0.0
for r in auair_results:
    rt_factor = r['duration_s'] / r['elapsed_s']
    total_dur += r['duration_s']
    total_elapsed += r['elapsed_s']
    print(f"{r['video_id']:30}{r['windows']:>9}{r['duration_s']:>10.1f}"
          f"{rt_factor:>13.2f}x{r['vehicle_total']:>7}")

agg_factor = total_dur / total_elapsed
print(f"\nToplam: {total_dur:.1f}s video, {total_elapsed:.1f}s isleme suresi")
print(f"AGREGE gercek-zaman katsayisi: {agg_factor:.3f}x")
print(f"proje-ozeti.md §8 varsayimi: 40x  |  fark: {40/agg_factor:.0f}x daha yavas"
      if agg_factor > 0 else "")

## 8. Sorgu testleri

`clips_auair_test` üzerinde gerçek sorgular. vLLM yok - yapısal ayrıştırma
devre dışı, sorgu tamamen semantiğe düşüyor (`vehicle_count` filtresi manuel
kurulmadan test edilemez, bu defterde o yol yok - genel notebook'ta var).

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

assert config.QDRANT_COLLECTION == 'clips_auair_test', (
    'Yanlis koleksiyon - 3. bolumdeki set_env calisti mi?')

AUAIR_PROMPTS = [
    'a truck at a roundabout',
    'an ambulance on the road',
    'cars waiting at an intersection',
    'a busy road with many vehicles',
    'a motorbike on the street',
]

for p in AUAIR_PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    render(run_query(p, top_k=5))
    print()

print('=' * 70)
print('BAKILACAK:')
print('  - Sonuclar mantikli mi (kavsak/trafik sahneleri donuyor mu)?')
print('  - video_id alani "auair_frame_..." formatinda mi (dogru koleksiyon)?')
print('  - agl_m/avg_speed_kmh degerleri fiziksel olarak makul mu '
      '(bkz. auair_adapter.py doc: ~5-30m, ~0-16 km/h)?')

## 9. Sonuçları kaydedin

Bu denemeden çıkan, proje-ozeti.md §8'e ve dataset kararına girecek sayılar:

1. **Agrege gerçek-zaman katsayısı** (7. bölümün özet tablosu) — yerel
   makinede ölçülen 0.10x ile karşılaştırın; T4'te ne kadar farklı çıkıyor,
   §8'in 40x varsayımına ne kadar yaklaşıyor.
2. **8 videonun tamamı gerçekten ingest edildi mi** — hata alan video oldu
   mu, oldu ise hangi adımda (proxy/telemetri/embedding/YOLO/yazım).
3. **Sorgu sonuçları mantıklı mı** — "a truck at a roundabout" gibi
   sorgular gerçekten kavşak/trafik sahneleri mi döndürüyor, yoksa alakasız
   mı (retrieval KALİTESİ testi değil ama tamamen rastgele de olmamalı).
4. **Telemetri alanları fiziksel olarak makul mü** — 8 videonun tamamında
   `agl_m`/`avg_speed_kmh` aralıkları `poc/auair_adapter.py`'nin docstring'inde
   belirtilen aralıklarla (4.8-30.2m, 0.05-16 km/h) tutarlı mı, yoksa bazı
   videolarda sapma var mı.

**Unutmayın:** bu deneme AU-AIR'in **CC BY-NC-SA/CC BY-NC** lisansı altında,
sadece iç doğrulama amaçlı. Sonuçlar "AU-AIR mekanizma testi olarak işe
yarıyor mu" sorusuna cevap veriyor — "hangi embedding modeli/pencereleme
daha iyi" sorusuna değil (bunun için golden set gerekiyor, §7 hâlâ boş).

Sonuçları bu notebook'un çıktısıyla birlikte paylaşın; `docs/` altına yeni
bir worklog girişi olarak işlenecek.